In [1]:
# Import các thư viện cơ bản
import pandas as pd
import os
import joblib

# Import các công cụ từ scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Đọc dữ liệu
duong_dan_file = "data/clean_data.csv"
df = pd.read_csv(duong_dan_file)

# Xóa các dòng bị lỗi trống (nếu có trong quá trình lưu file)
df = df.dropna(subset=['Cleaned_Review', 'Label'])

print("Tổng số dữ liệu sẵn sàng huấn luyện:", len(df))
df.head()

Tổng số dữ liệu sẵn sàng huấn luyện: 1985


,Review,Cleaned_Review,Label
0,"Tất dày dặn, đi êm chân. Giao hàng nhanh. Nên mua",tất_dày dặn êm_chân giao nhanh nên mua,1
1,"Cực kì cực kì hài lòng, rất đáng tiền...",cực_kì cực_kì hài_lòng rất đáng tiền giao nhan...,1
2,Đúng như giới thiệu,đúng như giới_thiệu,1
3,"Đẹp ,tốt nhận hàng thích",đẹp tốt nhận thích,1
4,"🔰🔰🔰 Nhà bán xử lý đơn hàng cực nhanh, bán hàng...",Japanese_symbol_for_beginer Japanese_symbol_fo...,1


In [2]:
from sklearn.model_selection import train_test_split

# Lấy tập văn bản làm đầu vào (X) và tập nhãn làm mục tiêu (y)
X = df['Cleaned_Review']
y = df['Label']

# Kiểm tra số lượng dữ liệu để quyết định cách chia
if len(X) < 50:
    print("[Cảnh báo] Dữ liệu hiện tại quá ít. Hệ thống sẽ chia ngẫu nhiên bình thường (Bỏ qua Stratify).")
    # Lần 1: Chia 70% Train, 30% Temp (Không dùng stratify)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    # Lần 2: Chia đôi Temp thành 15% Val và 15% Test
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

else:
    print("[OK] Dữ liệu đủ lớn. Đang tiến hành chia phân tầng (Stratify) theo chuẩn...")
    # Lần 1: Chia có stratify
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    # Lần 2: Chia có stratify
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("-" * 40)
print("Tổng số câu dữ liệu hiện có:", len(X))
print(f"Số câu tập Train (70%): {len(X_train)} câu")
print(f"Số câu tập Validation (15%): {len(X_val)} câu")
print(f"Số câu tập Test (15%): {len(X_test)} câu")

[OK] Dữ liệu đủ lớn. Đang tiến hành chia phân tầng (Stratify) theo chuẩn...
----------------------------------------
Tổng số câu dữ liệu hiện có: 1985
Số câu tập Train (70%): 1389 câu
Số câu tập Validation (15%): 298 câu
Số câu tập Test (15%): 298 câu


In [3]:
import time
from custom_tfidf import CustomTfidfVectorizer 

print("="*70)
print(" CHUYỂN HÓA VĂN BẢN THÀNH MA TRẬN (CUSTOM TF-IDF) ".center(70, "="))
print("="*70)

# 1. Khởi tạo công cụ TF-IDF
vectorizer = CustomTfidfVectorizer()

start_time = time.time()
print("[INFO] Đang xây dựng từ điển và biến đổi tập Train (fit_transform)...")
# 2. Học từ điển từ tập Train và chuyển nó thành ma trận số
X_train_vector = vectorizer.fit_transform(X_train)

print("[INFO] Đang áp dụng từ điển cho tập Validation và Test (transform)...")
# 3. Chỉ "áp dụng" (transform) từ điển đã học lên tập Val và Test để tránh Data Leakage
X_val_vector = vectorizer.transform(X_val)
X_test_vector = vectorizer.transform(X_test)

end_time = time.time()
print(f"[SUCCESS] Hoàn tất quá trình TF-IDF trong {round(end_time - start_time, 2)} giây!")
print(f"[METRIC] Kích thước bộ từ vựng (Số lượng features): {len(vectorizer.vocab)} từ.")
print("[INFO] Dữ liệu (Model-ready) đã sẵn sàng để đưa vào AI!")
print("="*70)

========== CHUYỂN HÓA VĂN BẢN THÀNH MA TRẬN (CUSTOM TF-IDF) ==========
[INFO] Đang xây dựng từ điển và biến đổi tập Train (fit_transform)...
[INFO] Đang áp dụng từ điển cho tập Validation và Test (transform)...
[SUCCESS] Hoàn tất quá trình TF-IDF trong 0.47 giây!
[METRIC] Kích thước bộ từ vựng (Số lượng features): 2641 từ.
[INFO] Dữ liệu (Model-ready) đã sẵn sàng để đưa vào AI!


In [4]:
import os
import joblib
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Import 2 thuật toán tự chế thay vì dùng thư viện có sẵn
from custom_naive_bayes import CustomMultinomialNB
from custom_logistic_regression import CustomLogisticRegression

print("ĐANG TỰ ĐỘNG HUẤN LUYỆN VÀ TÌM KIẾM MÔ HÌNH TỐT NHẤT...\n")

# Chuyển đổi Pandas Series sang Python list để tương thích với các Class tự chế
y_train_list = y_train.tolist() if hasattr(y_train, 'tolist') else y_train
y_val_list = y_val.tolist() if hasattr(y_val, 'tolist') else y_val

# 1. Huấn luyện Naive Bayes
mo_hinh_nb = CustomMultinomialNB(alpha=1.0, fit_prior=False)
mo_hinh_nb.fit(X_train_vector, y_train_list)
du_doan_nb = mo_hinh_nb.predict(X_val_vector)
# Dùng average='macro' để F1-score chấm điểm công bằng cho cả lớp Tích cực và Tiêu cực
f1_nb = f1_score(y_val_list, du_doan_nb, average='macro', zero_division=0) 

# 2. Huấn luyện Logistic Regression
# Lưu ý: Logistic học qua nhiều vòng lặp nên sẽ mất khoảng vài giây để chạy xong
mo_hinh_lr = CustomLogisticRegression(learning_rate=0.5, num_iterations=500, class_weight={0: 10.0, 1: 1.0})
mo_hinh_lr.fit(X_train_vector, y_train_list)
du_doan_lr = mo_hinh_lr.predict(X_val_vector)
f1_lr = f1_score(y_val_list, du_doan_lr, average='macro', zero_division=0)

print(f"[Naive Bayes]        Điểm F1-Score: {round(f1_nb * 100, 2)}%")
print(f"[Logistic Reg]      Điểm F1-Score: {round(f1_lr * 100, 2)}%\n")

# 3. TỰ ĐỘNG SO SÁNH VÀ LỰA CHỌN (AI TỰ QUYẾT ĐỊNH)
if f1_lr > f1_nb:
    print("=> TỰ ĐỘNG CHỌN: LOGISTIC REGRESSION làm mô hình chính thức!")
    best_model = mo_hinh_lr
    best_predict = du_doan_lr
    ten_mo_hinh = "Logistic Regression"
else:
    # Nếu điểm bằng nhau hoặc NB cao hơn, chọn NB
    print("=> TỰ ĐỘNG CHỌN: NAIVE BAYES làm mô hình chính thức!")
    best_model = mo_hinh_nb
    best_predict = du_doan_nb
    ten_mo_hinh = "Naive Bayes"

# 4. In báo cáo chi tiết cho mô hình chiến thắng
print("-" * 55)
print(f"BẢNG BÁO CÁO CHI TIẾT CỦA THUẬT TOÁN {ten_mo_hinh.upper()}:")
# zero_division=0 giúp triệt tiêu các cảnh báo lỗi khi dữ liệu quá ít
print(classification_report(y_val_list, best_predict, target_names=['Tiêu cực (0)', 'Tích cực (1)'], zero_division=0))
print("-" * 55)

# 5. Xuất file vật lý cho mô hình tốt nhất
thu_muc_luu = "saved_models"
if not os.path.exists(thu_muc_luu):
    os.makedirs(thu_muc_luu)

# Lưu Vectorizer (Từ điển)
joblib.dump(vectorizer, os.path.join(thu_muc_luu, "tfidf_vectorizer.pkl"))

# Lưu Mô hình tốt nhất vừa được chọn ở bước 3
joblib.dump(best_model, os.path.join(thu_muc_luu, "sentiment_model.pkl"))

print(f"[+] Hoàn tất! Hệ thống đã lưu bộ não của {ten_mo_hinh} vào thư mục saved_models.")

ĐANG TỰ ĐỘNG HUẤN LUYỆN VÀ TÌM KIẾM MÔ HÌNH TỐT NHẤT...

[Naive Bayes]        Điểm F1-Score: 57.84%
[Logistic Reg]      Điểm F1-Score: 71.78%

=> TỰ ĐỘNG CHỌN: LOGISTIC REGRESSION làm mô hình chính thức!
-------------------------------------------------------
BẢNG BÁO CÁO CHI TIẾT CỦA THUẬT TOÁN LOGISTIC REGRESSION:
              precision    recall  f1-score   support

Tiêu cực (0)       0.40      0.68      0.51        28
Tích cực (1)       0.96      0.90      0.93       270

    accuracy                           0.88       298
   macro avg       0.68      0.79      0.72       298
weighted avg       0.91      0.88      0.89       298

-------------------------------------------------------
[+] Hoàn tất! Hệ thống đã lưu bộ não của Logistic Regression vào thư mục saved_models.
